In [ ]:
!pip install pandas numpy matplotlib seaborn

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set up plotting for Colab
%matplotlib inline

# Upload the dataset
from google.colab import files
uploaded = files.upload()

# Load the dataset (assumes the file is named 'formatting_data.csv')
df = pd.read_csv('formatting_data.csv')

# Convert datetime columns
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')
df['First Interaction Date'] = pd.to_datetime(df['First Interaction Date'], errors='coerce')
df['Last Purchase Date'] = pd.to_datetime(df['Last Purchase Date'], errors='coerce')

print("Dataset loaded successfully!")

Saving formatting_data.csv to formatting_data.csv
Dataset loaded successfully!


## Dataset's Structure, Missing values, Key summaries, Trends and EDA

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set plotting style
sns.set(style="whitegrid")

def basic_eda(df):
    """
    Perform basic Exploratory Data Analysis on the DataFrame with an improved correlation matrix.

    Parameters:
    - df (pd.DataFrame): Input DataFrame to analyze
    """
    # Create 'eda_charts' folder if it doesn't exist
    if not os.path.exists('eda_charts'):
        os.makedirs('eda_charts')

    # Helper function to check column existence
    def has_columns(*cols):
        return all(col in df.columns for col in cols)

    # 1. Basic Information
    print("=== Basic Information ===")
    print(f"Shape of DataFrame: {df.shape}")
    print(f"\nColumns: {list(df.columns)}")
    print("\nData Types:")
    print(df.dtypes)

    # 2. Summary Statistics
    print("\n=== Summary Statistics ===")
    print(df.describe(include='all'))

    # 3. Missing Values
    print("\n=== Missing Values ===")
    missing_values = df.isnull().sum()
    missing_percent = (missing_values / len(df)) * 100
    missing_df = pd.DataFrame({'Missing Count': missing_values, 'Missing %': missing_percent})
    print(missing_df[missing_df['Missing Count'] > 0])

    # 4. Unique Values
    print("\n=== Unique Values per Column ===")
    unique_counts = df.nunique()
    print(unique_counts)

    # 5. Visualizations
    # Distribution of Numerical Columns
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
    for col in numeric_cols:
        if col in ['Total Revenue', 'Quantity Sold', 'Price Paid', 'AOV', 'COGS', 'Net Profit']:
            plt.figure(figsize=(8, 5))
            sns.histplot(df[col].dropna(), bins=30, kde=True)
            plt.title(f'Distribution of {col}')
            plt.xlabel(col)
            plt.ylabel('Frequency')
            plt.savefig(f'eda_charts/distribution_{col.lower().replace(" ", "_")}.png')
            plt.close()

    # Boxplot for Outliers
    for col in numeric_cols:
        if col in ['Total Revenue', 'Quantity Sold', 'Price Paid', 'AOV', 'COGS', 'Net Profit']:
            plt.figure(figsize=(8, 5))
            sns.boxplot(x=df[col].dropna())
            plt.title(f'Boxplot of {col}')
            plt.xlabel(col)
            plt.savefig(f'eda_charts/boxplot_{col.lower().replace(" ", "_")}.png')
            plt.close()

    # Categorical Column Distribution
    categorical_cols = df.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        if col in ['Category', 'Channel', 'Customer Segment', 'Gender', 'Region', 'Location']:
            plt.figure(figsize=(10, 6))
            sns.countplot(y=df[col], order=df[col].value_counts().index)
            plt.title(f'Distribution of {col}')
            plt.xlabel('Count')
            plt.ylabel(col)
            plt.savefig(f'eda_charts/countplot_{col.lower().replace(" ", "_")}.png')
            plt.close()

    # Time Series Plot (if Transaction Date exists)
    if has_columns('Transaction Date'):
        df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')
        if has_columns('Total Revenue'):
            revenue_over_time = df.groupby(df['Transaction Date'].dt.to_period('M'))['Total Revenue'].sum()
            plt.figure(figsize=(12, 6))
            revenue_over_time.plot(kind='line')
            plt.title('Revenue Over Time')
            plt.xlabel('Month')
            plt.ylabel('Total Revenue ($)')
            plt.savefig('eda_charts/revenue_over_time.png')
            plt.close()

        if has_columns('Quantity Sold'):
            units_over_time = df.groupby(df['Transaction Date'].dt.to_period('M'))['Quantity Sold'].sum()
            plt.figure(figsize=(12, 6))
            units_over_time.plot(kind='line')
            plt.title('Units Sold Over Time')
            plt.xlabel('Month')
            plt.ylabel('Quantity Sold')
            plt.savefig('eda_charts/units_sold_over_time.png')
            plt.close()

    # Improved Correlation Matrix
    if len(numeric_cols) > 1:
        # Select key numeric columns to avoid clutter
        key_numeric_cols = [col for col in numeric_cols if col in [
            'Total Revenue', 'Quantity Sold', 'Price Paid', 'AOV', 'Basket Size',
            'Cost per Acquisition (CPA)', 'COGS', 'Gross Profit', 'Net Profit', 'Operating Expenses'
        ]]

        if key_numeric_cols:
            corr_matrix = df[key_numeric_cols].corr()

            # Create a mask for the upper triangle
            mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

            # Set up the matplotlib figure
            plt.figure(figsize=(12, 10))

            # Draw the heatmap with improved styling
            sns.heatmap(
                corr_matrix,
                mask=mask,  # Hide upper triangle
                annot=True,  # Show correlation values
                fmt='.2f',  # 2 decimal places
                cmap='RdBu_r',  # Diverging color palette (red-blue)
                vmin=-1, vmax=1,  # Set range from -1 to 1
                center=0,  # Center at 0
                square=True,  # Square cells
                linewidths=0.5,  # Add grid lines
                cbar_kws={'shrink': .5, 'label': 'Correlation'},  # Customize colorbar
                annot_kws={'size': 10}  # Font size for annotations
            )

            # Customize labels and title
            plt.title('Correlation Matrix of Key Numeric Variables', fontsize=16, pad=20)
            plt.xticks(rotation=45, ha='right', fontsize=12)
            plt.yticks(rotation=0, fontsize=12)

            # Adjust layout to prevent clipping
            plt.tight_layout()

            # Save with higher resolution
            plt.savefig('eda_charts/correlation_matrix_improved.png', dpi=300, bbox_inches='tight')
            plt.close()
        else:
            print("No key numeric columns found for correlation matrix.")

    print("\nEDA complete. Visualizations saved in 'eda_charts' folder.")

# Run EDA
basic_eda(df)

=== Basic Information ===
Shape of DataFrame: (3000, 136)

Columns: ['Name', 'Email', 'Location', 'Zip Code', 'Age', 'Gender', 'Language', 'Loyalty Program Member', 'Product Name', 'Purchase History', 'Avg Order Value', 'Cart Abandoned', 'Payment Method', 'Page Views', 'Time Spent (min)', 'Clickstream Data', 'Search Query', 'Bounce Rate', 'Newsletter Opened', 'Product Rating', 'Support Queries', 'Survey Response', 'Social Media Share', 'Device Type', 'Browser Type', 'Operating System', 'IP Address', 'Referral Source', 'Promotional Elasticity', 'Loyalty Effect on Purchase', 'Engagement Score', 'Cost per Acquisition (CPA)', 'Cost per Channel', 'Total Revenue', 'Gross Profit', 'Net Profit', 'COGS', 'Operating Expenses', 'Profit Margin (%)', 'Revenue Trend', 'Expense Breakdown', 'Gross vs Net Profit', 'Region-wise Revenue Contribution', 'Churned', 'Customer ID', 'Transaction Date', 'Previous Category', 'Total Completed Purchase Value', 'Total Orders in Time Slot', 'Total Floated Coupon', '

# **Calculation For KPI's In Diagnostic Section**

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Create 'charts' folder if it doesn't exist
charts_folder = 'charts'
if not os.path.exists(charts_folder):
    os.makedirs(charts_folder)

# Data validation function
def validate_data(df, required_columns):
    missing_cols = [col for col in required_columns if col not in df.columns]
    if missing_cols:
        print(f"Warning: Missing columns: {missing_cols}")
    for col in df.columns:
        if df[col].isnull().sum() > 0:
            print(f"Warning: {col} has {df[col].isnull().sum()} missing values")
        if df[col].dtype in ['float64', 'int64'] and (df[col] < 0).any():
            print(f"Warning: {col} has negative values")

# Helper functions with caps
def calculate_drop_off_rate(users_n, users_n_plus_1):
    if users_n == 0:
        return 0
    return max(0, min(100, ((users_n - users_n_plus_1) / users_n) * 100))

def calculate_conversion_rate(users_n_plus_1, users_n):
    if users_n == 0:
        return 0
    return max(0, min(100, (users_n_plus_1 / users_n) * 100))

# Define required columns
required_columns = [
    'Cart Abandoned', 'Checkout Initiated', 'Payments Successful', 'Payments Attempted',
    'Bounce Rate', 'Awareness Stage', 'Consideration Stage', 'Intent Stage', 'Purchase Stage',
    'Device Type', 'Loyalty Program Member', 'Churned', 'Transaction Date', 'Customer ID',
    'Repeat Purchase Rate', 'Region', 'Churn Reason', 'Reactivated Customers (Per Campaign)',
    'Subscription Cancelled', 'Discount Applied', 'Total Floated Coupon', 'Product Name',
    'Add to Cart Count', 'Product Price', 'Quantity Sold', 'Return Requests',
    'Out of Stock Occurrences', 'Support Queries', 'Defective Complaints', 'Stock Availability',
    'Channel', 'Order Fulfillment Time', 'Survey Response', 'Page Views', 'Ad Clicks',
    'Impressions', 'Newsletter Opened', 'Referral Source', 'Delivery Time', 'Delivery Delays',
    'Reason for Delivery Delays', 'Order Shipment Time', 'Total Units', 'Return Due to Delivery Issues',
    'Number of Chargebacks', 'Reasons for Payment Failure', 'Payment Method',
    'Average Response Time', 'Ticket Resolution Time', 'Engagement Score', 'CLV'
]

# Validate data
print("Validating dataset...")
validate_data(df, required_columns)
print("\nCalculating KPIs and saving charts...\n")

# Validate data
print("Validating dataset...")
validate_data(df, required_columns)
print("\nCalculating KPIs and saving charts...\n")

# Cache common aggregations
cache = {}
if 'Transaction Date' in df.columns:
    df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')
    cache['by_date'] = df.groupby('Transaction Date')
if 'Customer ID' in df.columns:
    cache['by_customer'] = df.groupby('Customer ID')
if 'Product Name' in df.columns:
    cache['by_product'] = df.groupby('Product Name')

# Preprocess data
if 'Bounce Rate' in df.columns:
    df['Bounce Rate'] = df['Bounce Rate'].apply(lambda x: x / 100 if x > 1 else x)
binary_cols = ['Cart Abandoned', 'Churned', 'Subscription Cancelled', 'Loyalty Program Member']
for col in binary_cols:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: 1 if x > 0 else 0)
stages = ['Awareness Stage', 'Consideration Stage', 'Intent Stage', 'Purchase Stage']

# Convert timedeltas and numeric columns
timedelta_cols = ['Average Response Time', 'Ticket Resolution Time', 'Delivery Time', 'Order Fulfillment Time', 'Order Shipment Time']
for col in timedelta_cols:
    if col in df.columns:
        df[col] = pd.to_timedelta(df[col], errors='coerce')
if 'Discount Applied' in df.columns:
    df['Discount Applied'] = pd.to_numeric(df['Discount Applied'].replace(r'[%]', '', regex=True), errors='coerce').fillna(0)
if 'Total Floated Coupon' in df.columns:
    df['Total Floated Coupon'] = pd.to_numeric(df['Total Floated Coupon'].replace(r'[%]', '', regex=True), errors='coerce').fillna(0)

# 1️⃣ Customer Journey Analysis (Drop-off & Abandonment)
print("1️⃣ Customer Journey Analysis (Drop-off & Abandonment)")
print("Card KPIs:")
# Cart Abandonment Rate
if 'Cart Abandoned' in df.columns:
    carts_created = len(df[df['Cart Abandoned'].notnull()])
    carts_abandoned = df['Cart Abandoned'].sum()
    cart_abandonment_rate = min(100, (carts_abandoned / carts_created) * 100) if carts_created > 0 else 0
    print(f"Cart Abandonment Rate: {cart_abandonment_rate:.2f}%")
else:
    print("Cart Abandonment Rate: Skipped (missing 'Cart Abandoned')")

# Checkout Abandonment Rate
if all(col in df.columns for col in ['Checkout Initiated', 'Payments Successful']):
    carts_reached_checkout = df['Checkout Initiated'].sum()
    completed_purchases = df['Payments Successful'].sum()
    checkout_abandonment_rate = min(100, ((carts_reached_checkout - completed_purchases) / carts_reached_checkout) * 100) if carts_reached_checkout > 0 else 0
    print(f"Checkout Abandonment Rate: {checkout_abandonment_rate:.2f}%")
else:
    print("Checkout Abandonment Rate: Skipped (missing required columns)")

# Payment Failure Rate
if all(col in df.columns for col in ['Payments Attempted', 'Payments Successful']):
    failed_payments = df['Payments Attempted'].sum() - df['Payments Successful'].sum()
    total_payment_attempts = df['Payments Attempted'].sum()
    payment_failure_rate = min(100, (failed_payments / total_payment_attempts) * 100) if total_payment_attempts > 0 else 0
    print(f"Payment Failure Rate: {payment_failure_rate:.2f}%")
else:
    print("Payment Failure Rate: Skipped (missing required columns)")

# Product Page Bounce Rate
if 'Bounce Rate' in df.columns:
    product_page_bounce_rate = df['Bounce Rate'].mean() * 100
    print(f"Product Page Bounce Rate: {product_page_bounce_rate:.2f}%")
else:
    print("Product Page Bounce Rate: Skipped (missing 'Bounce Rate')")

# Drop-off Trend Over Time (Line Chart)
if all(stage in df.columns for stage in stages):
    drop_off_trend = {}
    for i in range(len(stages) - 1):
        users_n = df[stages[i]].sum()
        users_n_plus_1 = df[stages[i + 1]].sum()
        drop_off_rate = calculate_drop_off_rate(users_n, users_n_plus_1)
        drop_off_trend[f"{stages[i]} to {stages[i + 1]}"] = drop_off_rate
    plt.figure(figsize=(10, 6))
    plt.plot(list(drop_off_trend.keys()), list(drop_off_trend.values()), marker='o')
    plt.title("Drop-off Trend Over Time")
    plt.xlabel("Funnel Stages")
    plt.ylabel("Drop-off Rate (%)")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '1_drop_off_trend_over_time.png'))
    plt.close()
else:
    print("Drop-off Trend Over Time: Skipped (missing stage columns)")

# Drop-off by Customer Segment (Heatmap)
segment_column = 'Loyalty Program Member'
if segment_column in df.columns and all(stage in df.columns for stage in stages):
    segments = df[segment_column].unique()
    heatmap_data = pd.DataFrame(index=segments, columns=[f"{stages[i]} to {stages[i+1]}" for i in range(len(stages)-1)])
    for segment in segments:
        segment_df = df[df[segment_column] == segment]
        for i in range(len(stages) - 1):
            users_n = segment_df[stages[i]].sum()
            users_n_plus_1 = segment_df[stages[i + 1]].sum()
            drop_off_rate = calculate_drop_off_rate(users_n, users_n_plus_1)
            heatmap_data.loc[segment, f"{stages[i]} to {stages[i+1]}"] = drop_off_rate
    plt.figure(figsize=(10, 6))
    sns.heatmap(heatmap_data.astype(float), annot=True, cmap="YlGnBu", fmt=".1f")
    plt.title("Drop-off by Customer Segment")
    plt.xlabel("Funnel Stages")
    plt.ylabel("Segment")
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '1_drop_off_by_customer_segment.png'))
    plt.close()
else:
    print("Drop-off by Customer Segment: Skipped (missing required columns)")

# Conversion Funnel Analysis (Bar Chart)
if all(stage in df.columns for stage in stages):
    conversion_funnel = [df[stage].sum() for stage in stages]
    plt.figure(figsize=(10, 6))
    plt.bar(stages, conversion_funnel, color='skyblue')
    plt.title("Conversion Funnel Analysis")
    plt.xlabel("Funnel Stage")
    plt.ylabel("Users")
    for i, v in enumerate(conversion_funnel):
        plt.text(i, v + 0.5, str(int(v)), ha='center')
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '1_conversion_funnel_analysis.png'))
    plt.close()
else:
    print("Conversion Funnel Analysis: Skipped (missing stage columns)")

# Cart Abandonment by Device Type (Stacked Bar Chart)
if all(col in df.columns for col in ['Device Type', 'Cart Abandoned']):
    abandonment_by_device = df.groupby('Device Type')['Cart Abandoned'].agg(['sum', 'count'])
    abandonment_by_device['Abandoned'] = abandonment_by_device['sum']
    abandonment_by_device['Not Abandoned'] = abandonment_by_device['count'] - abandonment_by_device['sum']
    plt.figure(figsize=(10, 6))
    abandonment_by_device[['Abandoned', 'Not Abandoned']].plot(kind='bar', stacked=True, color=['red', 'green'])
    plt.title("Cart Abandonment by Device Type")
    plt.xlabel("Device Type")
    plt.ylabel("Count")
    plt.legend(title="Status")
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '1_cart_abandonment_by_device_type.png'))
    plt.close()
else:
    print("Cart Abandonment by Device Type: Skipped (missing required columns)")

# 2️⃣ Churn & Retention Analysis
print("\n2️⃣ Churn & Retention Analysis")
print("Card KPIs:")
# Churn Rate by Consumer Segment
if all(col in df.columns for col in ['Loyalty Program Member', 'Churned']):
    churn_rate_by_segment = df.groupby('Loyalty Program Member')['Churned'].mean() * 100
    print("Churn Rate by Consumer Segment:", churn_rate_by_segment.to_dict())
else:
    print("Churn Rate by Consumer Segment: Skipped (missing required columns)")

# Avg. Time Between Purchases
if all(col in df.columns for col in ['Transaction Date', 'Customer ID']):
    customer_purchases = cache['by_customer']['Transaction Date'].agg(['min', 'max', 'count'])
    multi_purchase = customer_purchases[customer_purchases['count'] > 1]
    if not multi_purchase.empty:
        avg_days = (multi_purchase['max'] - multi_purchase['min']).dt.days / (multi_purchase['count'] - 1)
        avg_time_between_purchases = avg_days.mean() if not avg_days.empty else 0
    else:
        avg_time_between_purchases = 0
    print(f"Avg. Time Between Purchases: {avg_time_between_purchases:.2f} days")
else:
    print("Avg. Time Between Purchases: Skipped (missing required columns)")

# Repeat Purchase Decline (%)
if 'Repeat Purchase Rate' in df.columns and 'Transaction Date' in df.columns:
    repeat_purchase_trend = cache['by_date']['Repeat Purchase Rate'].mean()
    if len(repeat_purchase_trend) > 1:
        repeat_purchase_start = repeat_purchase_trend.iloc[0]
        repeat_purchase_end = repeat_purchase_trend.iloc[-1]
        repeat_purchase_decline = ((repeat_purchase_start - repeat_purchase_end) / repeat_purchase_start) * 100 if repeat_purchase_start > 0 else 0
    else:
        repeat_purchase_decline = 0
    print(f"Repeat Purchase Decline: {repeat_purchase_decline:.2f}%")
else:
    print("Repeat Purchase Decline: Skipped (missing required columns)")

# Loyalty Program Drop-off Rate
if all(col in df.columns for col in ['Loyalty Program Member', 'Churned']):
    loyalty_customers = df[df['Loyalty Program Member'] == 1]
    total_loyalty_customers = len(loyalty_customers)
    loyalty_customers_lost = loyalty_customers['Churned'].sum()
    loyalty_drop_off_rate = min(100, (loyalty_customers_lost / total_loyalty_customers) * 100) if total_loyalty_customers > 0 else 0
    print(f"Loyalty Program Drop-off Rate: {loyalty_drop_off_rate:.2f}%")
else:
    print("Loyalty Program Drop-off Rate: Skipped (missing required columns)")

# Net Promoter Score (NPS) - Added from prior discussion
if 'Survey Response' in df.columns:
    survey_scores = df['Survey Response'].dropna()
    promoters = survey_scores[survey_scores.isin([9, 10])].count() if pd.api.types.is_numeric_dtype(survey_scores) else survey_scores.str.contains('Positive', case=False).sum()
    detractors = survey_scores[survey_scores.isin([0, 1, 2, 3, 4, 5, 6])].count() if pd.api.types.is_numeric_dtype(survey_scores) else survey_scores.str.contains('Negative', case=False).sum()
    total_responses = survey_scores.count()
    nps = ((promoters / total_responses) - (detractors / total_responses)) * 100 if total_responses > 0 else 0
    print(f"Net Promoter Score (NPS): {nps:.2f}")
else:
    print("Net Promoter Score (NPS): Skipped (missing 'Survey Response')")

# Retention Rate by Region (Stacked Bar Chart)
if all(col in df.columns for col in ['Region', 'Churned']):
    retention_by_region = df.groupby('Region')['Churned'].agg(['sum', 'count'])
    retention_by_region['Retained'] = retention_by_region['count'] - retention_by_region['sum']
    retention_by_region['Churned'] = retention_by_region['sum']
    plt.figure(figsize=(10, 6))
    retention_by_region[['Retained', 'Churned']].plot(kind='bar', stacked=True, color=['green', 'red'])
    plt.title("Retention Rate by Region")
    plt.xlabel("Region")
    plt.ylabel("Count")
    plt.legend(title="Status")
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '2_retention_rate_by_region.png'))
    plt.close()
else:
    print("Retention Rate by Region: Skipped (missing required columns)")

# Churn Reasons (Bar Chart)
if all(col in df.columns for col in ['Churn Reason', 'Churned']):
    churn_reasons = df[df['Churned'] == 1]['Churn Reason'].value_counts()
    plt.figure(figsize=(10, 6))
    churn_reasons.plot(kind='bar', color='purple')
    plt.title("Churn Reasons (Survey-Based)")
    plt.xlabel("Reason")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '2_churn_reasons.png'))
    plt.close()
else:
    print("Churn Reasons: Skipped (missing required columns)")

# Win-Back Campaign Effectiveness (Column Chart)
if all(col in df.columns for col in ['Reactivated Customers (Per Campaign)', 'Churned']):
    reactivated_customers = df['Reactivated Customers (Per Campaign)'].sum()
    targeted_churned = len(df[df['Churned'] == 1])
    win_back_rate = min(100, (reactivated_customers / targeted_churned) * 100) if targeted_churned > 0 else 0
    plt.figure(figsize=(6, 4))
    plt.bar(['Win-Back Rate'], [win_back_rate], color='green')
    plt.title("Win-Back Campaign Effectiveness")
    plt.ylabel("Rate (%)")
    plt.ylim(0, 100)
    plt.text(0, win_back_rate + 2, f"{win_back_rate:.2f}%", ha='center')
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '2_win_back_campaign_effectiveness.png'))
    plt.close()
else:
    print("Win-Back Campaign Effectiveness: Skipped (missing required columns)")

# Subscription Cancellation Trend (Line Chart)
if all(col in df.columns for col in ['Transaction Date', 'Subscription Cancelled']):
    cancellation_trend = cache['by_date']['Subscription Cancelled'].mean() * 100
    plt.figure(figsize=(10, 6))
    cancellation_trend.plot(kind='line', marker='o', color='orange')
    plt.title("Subscription Cancellation Trend")
    plt.xlabel("Date")
    plt.ylabel("Cancellation Rate (%)")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '2_subscription_cancellation_trend.png'))
    plt.close()
else:
    print("Subscription Cancellation Trend: Skipped (missing required columns)")

# 3️⃣ Pricing Sensitivity & Discount Impact
print("\n3️⃣ Pricing Sensitivity & Discount Impact")
print("Card KPIs:")
# Avg. Discount Utilization Rate
if all(col in df.columns for col in ['Discount Applied', 'Total Floated Coupon']):
    discounts_used = df['Discount Applied'].sum()
    discounts_offered = df['Total Floated Coupon'].sum()
    avg_discount_utilization_rate = min(100, (discounts_used / discounts_offered) * 100) if discounts_offered > 0 else 0
    print(f"Avg. Discount Utilization Rate: {avg_discount_utilization_rate:.2f}%")
else:
    print("Avg. Discount Utilization Rate: Skipped (missing required columns)")

# Price Sensitivity Index - Newly calculated
if all(col in df.columns for col in ['Product Price', 'Quantity Sold', 'Transaction Date']):
    df['Month'] = df['Transaction Date'].dt.to_period('M')
    monthly_data = df.groupby(['Month', 'Product Name']).agg({'Product Price': 'mean', 'Quantity Sold': 'sum'}).reset_index()
    monthly_data['Price_Change'] = monthly_data.groupby('Product Name')['Product Price'].pct_change()
    monthly_data['Qty_Change'] = monthly_data.groupby('Product Name')['Quantity Sold'].pct_change()
    monthly_data['Price_Sensitivity'] = monthly_data['Qty_Change'] / monthly_data['Price_Change']
    price_sensitivity = monthly_data['Price_Sensitivity'].replace([np.inf, -np.inf]).mean()
    print(f"Price Sensitivity Index: {price_sensitivity:.2f}")
else:
    print("Price Sensitivity Index: Skipped (missing required columns)")

# Elasticity of Demand - Newly calculated
if all(col in df.columns for col in ['Product Price', 'Quantity Sold', 'Transaction Date']):
    elasticity = abs(price_sensitivity) if 'price_sensitivity' in locals() else 0
    print(f"Elasticity of Demand: {elasticity:.2f}")
else:
    print("Elasticity of Demand: Skipped (missing required columns)")

# Most Abandoned Discounted Product
if all(col in df.columns for col in ['Discount Applied', 'Product Name', 'Cart Abandoned', 'Add to Cart Count']):
    discounted_products = df[df['Discount Applied'] > 0]
    abandoned_products = discounted_products.groupby('Product Name')['Cart Abandoned'].sum()
    viewed_products = discounted_products.groupby('Product Name')['Add to Cart Count'].sum()
    product_abandonment_rate = (abandoned_products / viewed_products) * 100
    max_product = product_abandonment_rate.idxmax() if not product_abandonment_rate.empty else "N/A"
    max_rate = product_abandonment_rate.max() if not product_abandonment_rate.empty else 0
    print(f"Most Abandoned Discounted Product: {max_product} ({max_rate:.2f}%)")
else:
    print("Most Abandoned Discounted Product: Skipped (missing required columns)")

# Discounted vs. Non-Discounted Sales (Bar Chart)
if all(col in df.columns for col in ['Discount Applied', 'Quantity Sold', 'Product Price']):
    discounted_sales = df[df['Discount Applied'] > 0].apply(
        lambda x: x['Quantity Sold'] * (x['Product Price'] - (x['Product Price'] * x['Discount Applied'] / 100)), axis=1
    ).sum()
    non_discounted_sales = df[df['Discount Applied'] == 0].apply(
        lambda x: x['Quantity Sold'] * x['Product Price'], axis=1
    ).sum()
    plt.figure(figsize=(8, 6))
    plt.bar(['Discounted', 'Non-Discounted'], [discounted_sales, non_discounted_sales], color=['blue', 'orange'])
    plt.title("Discounted vs. Non-Discounted Sales")
    plt.ylabel("Sales Amount")
    for i, v in enumerate([discounted_sales, non_discounted_sales]):
        plt.text(i, v + 0.5, f"{v:.2f}", ha='center')
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '3_discounted_vs_non_discounted_sales.png'))
    plt.close()
else:
    print("Discounted vs. Non-Discounted Sales: Skipped (missing required columns)")

# Discount Effect on Repeat Purchase (Line Chart)
if all(col in df.columns for col in ['Transaction Date', 'Discount Applied', 'Repeat Purchase Rate']):
    discount_effect = cache['by_date'].apply(
        lambda x: x[x['Discount Applied'] > 0]['Repeat Purchase Rate'].mean() if not x[x['Discount Applied'] > 0].empty else 0
    )
    plt.figure(figsize=(10, 6))
    discount_effect.plot(kind='line', marker='o', color='teal')
    plt.title("Discount Effect on Repeat Purchase")
    plt.xlabel("Date")
    plt.ylabel("Repeat Purchase Rate")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '3_discount_effect_on_repeat_purchase.png'))
    plt.close()
else:
    print("Discount Effect on Repeat Purchase: Skipped (missing required columns)")

# Price vs. Conversion Rate (Scatter Plot)
if all(col in df.columns for col in ['Product Price', 'Payments Successful', 'Page Views']):
    price_vs_conversion = df.groupby('Product Price').apply(
        lambda x: (x['Payments Successful'].sum() / x['Page Views'].sum()) * 100 if x['Page Views'].sum() > 0 else 0
    )
    plt.figure(figsize=(10, 6))
    plt.scatter(price_vs_conversion.index, price_vs_conversion.values, color='purple')
    plt.title("Price vs. Conversion Rate")
    plt.xlabel("Product Price")
    plt.ylabel("Conversion Rate (%)")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '3_price_vs_conversion_rate.png'))
    plt.close()
else:
    print("Price vs. Conversion Rate: Skipped (missing required columns)")

# Discount Redemption Trend (Column Chart)
if all(col in df.columns for col in ['Transaction Date', 'Discount Applied', 'Total Floated Coupon']):
    redemption_trend = cache['by_date'].apply(
        lambda x: (x['Discount Applied'].sum() / x['Total Floated Coupon'].sum()) * 100 if x['Total Floated Coupon'].sum() > 0 else 0
    )
    plt.figure(figsize=(10, 6))
    redemption_trend.plot(kind='bar', color='coral')
    plt.title("Discount Redemption Trend")
    plt.xlabel("Date")
    plt.ylabel("Redemption Rate (%)")
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '3_discount_redemption_trend.png'))
    plt.close()
else:
    print("Discount Redemption Trend: Skipped (missing required columns)")

# 4️⃣ Customer Satisfaction & Support Issues
print("\n4️⃣ Customer Satisfaction & Support Issues")
print("Card KPIs:")
# Avg. Response Time (Customer Support)
if 'Average Response Time' in df.columns:
    avg_response_time = df['Average Response Time'].mean().total_seconds() / 3600 if df['Average Response Time'].notnull().any() else 0
    print(f"Avg. Response Time (hours): {avg_response_time:.2f}")
else:
    print("Avg. Response Time: Skipped (missing 'Average Response Time')")

# Avg. Resolution Time (Support Tickets)
if 'Ticket Resolution Time' in df.columns:
    avg_resolution_time = df['Ticket Resolution Time'].mean().total_seconds() / 3600 if df['Ticket Resolution Time'].notnull().any() else 0
    print(f"Avg. Resolution Time (hours): {avg_resolution_time:.2f}")
else:
    print("Avg. Resolution Time: Skipped (missing 'Ticket Resolution Time')")

# % of Unresolved Complaints - Fixed
if all(col in df.columns for col in ['Support Queries', 'Ticket Resolution Time']):
    unresolved_threshold = pd.Timedelta(days=7)  # Assume >7 days is unresolved
    unresolved_complaints = df[df['Ticket Resolution Time'] > unresolved_threshold]['Support Queries'].count()
    total_complaints = df['Support Queries'].count()
    unresolved_complaints_percentage = min(100, (unresolved_complaints / total_complaints) * 100) if total_complaints > 0 else 0
    print(f"% of Unresolved Complaints: {unresolved_complaints_percentage:.2f}%")
else:
    print("% of Unresolved Complaints: Skipped (missing required columns)")

# Most Common Complaint Category
if 'Support Queries' in df.columns:
    most_common_complaint = df['Support Queries'].value_counts().idxmax() if not df['Support Queries'].value_counts().empty else "N/A"
    print(f"Most Common Complaint Category: {most_common_complaint}")
else:
    print("Most Common Complaint Category: Skipped (missing 'Support Queries')")

# Ticket Resolution Time Trend (Line Chart)
if all(col in df.columns for col in ['Transaction Date', 'Ticket Resolution Time']):
    resolution_time_trend = cache['by_date']['Ticket Resolution Time'].mean().apply(
        lambda x: x.total_seconds() / 3600 if pd.notnull(x) else 0
    )
    plt.figure(figsize=(10, 6))
    resolution_time_trend.plot(kind='line', marker='o', color='navy')
    plt.title("Ticket Resolution Time Trend")
    plt.xlabel("Date")
    plt.ylabel("Resolution Time (hours)")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '4_ticket_resolution_time_trend.png'))
    plt.close()
else:
    print("Ticket Resolution Time Trend: Skipped (missing required columns)")

# Complaint Category Distribution (Pie Chart)
if 'Support Queries' in df.columns:
    complaint_distribution = df['Support Queries'].value_counts()
    plt.figure(figsize=(8, 8))
    plt.pie(complaint_distribution, labels=complaint_distribution.index, autopct='%1.1f%%', colors=sns.color_palette('pastel'))
    plt.title("Complaint Category Distribution")
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '4_complaint_category_distribution.png'))
    plt.close()
else:
    print("Complaint Category Distribution: Skipped (missing 'Support Queries')")

# Customer Satisfaction Score Trend (Stacked Area Chart)
if all(col in df.columns for col in ['Transaction Date', 'Survey Response']):
    csat_trend = cache['by_date']['Survey Response'].value_counts().unstack(fill_value=0)
    plt.figure(figsize=(10, 6))
    csat_trend.plot(kind='area', stacked=True, alpha=0.6)
    plt.title("Customer Satisfaction Score Trend")
    plt.xlabel("Date")
    plt.ylabel("Count")
    plt.legend(title="Response")
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '4_customer_satisfaction_score_trend.png'))
    plt.close()
else:
    print("Customer Satisfaction Score Trend: Skipped (missing required columns)")

# Impact of Support Quality on Repeat Purchase (Heatmap)
if all(col in df.columns for col in ['Ticket Resolution Time', 'Repeat Purchase Rate', 'Loyalty Program Member']):
    support_impact = df.groupby(['Loyalty Program Member', pd.cut(df['Ticket Resolution Time'], bins=5)])['Repeat Purchase Rate'].mean().unstack()
    plt.figure(figsize=(10, 6))
    sns.heatmap(support_impact, annot=True, cmap="YlGnBu", fmt=".2f")
    plt.title("Impact of Support Quality on Repeat Purchase")
    plt.xlabel("Resolution Time Bins")
    plt.ylabel("Loyalty Program Member")
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '4_impact_of_support_quality_on_repeat_purchase.png'))
    plt.close()
else:
    print("Impact of Support Quality on Repeat Purchase: Skipped (missing required columns)")

# 5️⃣ Product Issues & Performance Analysis
print("\n5️⃣ Product Issues & Performance Analysis")
print("Card KPIs:")
# Products with Highest Return Rate
if all(col in df.columns for col in ['Product Name', 'Return Requests', 'Quantity Sold']):
    return_rates = cache['by_product'].apply(
        lambda x: min(100, (x['Return Requests'].sum() / x['Quantity Sold'].sum()) * 100) if x['Quantity Sold'].sum() > 0 else 0
    )
    max_product = return_rates.idxmax() if not return_rates.empty else "N/A"
    max_rate = return_rates.max() if not return_rates.empty else 0
    print(f"Products with Highest Return Rate: {max_product} ({max_rate:.2f}%)")
else:
    print("Products with Highest Return Rate: Skipped (missing required columns)")

# Top Underperforming Products - Newly calculated
if all(col in df.columns for col in ['Product Name', 'Quantity Sold', 'Return Requests']):
    product_performance = cache['by_product'].agg({'Quantity Sold': 'sum', 'Return Requests': 'sum'})
    product_performance['Return_Rate'] = product_performance['Return Requests'] / product_performance['Quantity Sold']
    underperforming = product_performance[product_performance['Quantity Sold'] < product_performance['Quantity Sold'].quantile(0.25)]
    top_underperforming = underperforming.sort_values('Return_Rate', ascending=False).head(5)
    print("Top Underperforming Products:", top_underperforming[['Quantity Sold', 'Return_Rate']].to_dict())
else:
    print("Top Underperforming Products: Skipped (missing required columns)")

# Out-of-Stock Rate
if 'Out of Stock Occurrences' in df.columns:
    stockout_events = df['Out of Stock Occurrences'].sum()
    total_stock_events = len(df)
    out_of_stock_rate = min(100, (stockout_events / total_stock_events) * 100) if total_stock_events > 0 else 0
    print(f"Out-of-Stock Rate: {out_of_stock_rate:.2f}%")
else:
    print("Out-of-Stock Rate: Skipped (missing 'Out of Stock Occurrences')")

# Most Reported Product Issue
if 'Support Queries' in df.columns:
    most_reported_issue = df['Support Queries'].value_counts().idxmax() if not df['Support Queries'].value_counts().empty else "N/A"
    print(f"Most Reported Product Issue: {most_reported_issue}")
else:
    print("Most Reported Product Issue: Skipped (missing 'Support Queries')")

# Return Rate by Product Category (Bar Chart)
if all(col in df.columns for col in ['Category', 'Return Requests', 'Quantity Sold']):
    return_rate_by_category = df.groupby('Category').apply(
        lambda x: min(100, (x['Return Requests'].sum() / x['Quantity Sold'].sum()) * 100) if x['Quantity Sold'].sum() > 0 else 0
    )
    plt.figure(figsize=(10, 6))
    return_rate_by_category.plot(kind='bar', color='salmon')
    plt.title("Return Rate by Product Category")
    plt.xlabel("Category")
    plt.ylabel("Return Rate (%)")
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '5_return_rate_by_product_category.png'))
    plt.close()
else:
    print("Return Rate by Product Category: Skipped (missing required columns)")

# Stock-Out Events Over Time (Line Chart)
if all(col in df.columns for col in ['Transaction Date', 'Out of Stock Occurrences']):
    stock_out_trend = cache['by_date']['Out of Stock Occurrences'].sum()
    plt.figure(figsize=(10, 6))
    stock_out_trend.plot(kind='line', marker='o', color='darkred')
    plt.title("Stock-Out Events Over Time")
    plt.xlabel("Date")
    plt.ylabel("Stock-Out Events")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '5_stock_out_events_over_time.png'))
    plt.close()
else:
    print("Stock-Out Events Over Time: Skipped (missing required columns)")

# Defective Product Complaints (Stacked Area Chart)
if all(col in df.columns for col in ['Transaction Date', 'Defective Complaints', 'Product Name']):
    defective_trend = df.groupby(['Transaction Date', 'Product Name'])['Defective Complaints'].sum().unstack(fill_value=0)
    plt.figure(figsize=(10, 6))
    defective_trend.plot(kind='area', stacked=True, alpha=0.5)
    plt.title("Defective Product Complaints")
    plt.xlabel("Date")
    plt.ylabel("Complaints")
    plt.legend(title="Product Name", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '5_defective_product_complaints.png'))
    plt.close()
else:
    print("Defective Product Complaints: Skipped (missing required columns)")

# Demand-Supply Gap Analysis (Sankey or Bar Chart)
if all(col in df.columns for col in ['Quantity Sold', 'Stock Availability']):
    demand = df['Quantity Sold'].sum()
    supply = df[df['Stock Availability'] == 'In Stock']['Quantity Sold'].sum()
    gap = demand - supply
    if sankey and gap > 0:
        plt.figure(figsize=(10, 6))
        sankey(
            ['Demand', 'Supply', 'Gap'],
            ['Demand', 'Supply', 'Gap'],
            [demand, supply, gap],
            ['Demand to Supply', 'Demand to Gap', 'Supply'],
            colorDict={'Demand': 'red', 'Supply': 'green', 'Gap': 'gray'}
        )
        plt.title("Demand-Supply Gap Analysis")
        plt.tight_layout()
        plt.savefig(os.path.join(charts_folder, '5_demand_supply_gap_analysis.png'))
        plt.close()
    else:
        plt.figure(figsize=(8, 6))
        plt.bar(['Demand', 'Supply', 'Gap'], [demand, supply, gap], color=['red', 'green', 'gray'])
        plt.title("Demand-Supply Gap Analysis")
        plt.ylabel("Units")
        for i, v in enumerate([demand, supply, gap]):
            plt.text(i, v + 0.5, str(int(v)), ha='center')
        plt.tight_layout()
        plt.savefig(os.path.join(charts_folder, '5_demand_supply_gap_analysis.png'))
        plt.close()
else:
    print("Demand-Supply Gap Analysis: Skipped (missing required columns)")

# 6️⃣ Customer Segmentation & Engagement Analysis
print("\n6️⃣ Customer Segmentation & Engagement Analysis")
print("Card KPIs:")
# Avg. Engagement Score
if 'Engagement Score' in df.columns:
    avg_engagement_score = df['Engagement Score'].mean()
    print(f"Avg. Engagement Score: {avg_engagement_score:.2f}")
else:
    print("Avg. Engagement Score: Skipped (missing 'Engagement Score')")

# Customer Satisfaction by Segment
if all(col in df.columns for col in ['Loyalty Program Member', 'Survey Response']):
    satisfaction_by_segment = df.groupby('Loyalty Program Member')['Survey Response'].apply(
        lambda x: (x.str.contains('Positive', case=False, na=False).mean() * 100) if x.dtype == "object" else (x >= 9).mean() * 100
    )
    print("Customer Satisfaction by Segment:", satisfaction_by_segment.to_dict())
else:
    print("Customer Satisfaction by Segment: Skipped (missing required columns)")

# Customer Lifetime Value (CLV)
if 'CLV' in df.columns:
    avg_clv = df['CLV'].mean()
    print(f"Customer Lifetime Value (CLV): {avg_clv:.2f}")
else:
    print("Customer Lifetime Value (CLV): Skipped (missing 'CLV')")

# Customer Acquisition Cost (CAC) - Added
if all(col in df.columns for col in ['Marketing Cost', 'Customer ID']):
    total_marketing_cost = df['Marketing Cost'].sum()
    new_customers = df['Customer ID'].nunique()
    cac = total_marketing_cost / new_customers if new_customers > 0 else 0
    print(f"Customer Acquisition Cost (CAC): ${cac:.2f}")
else:
    print("Customer Acquisition Cost (CAC): Skipped (missing required columns)")

# Loyalty Program Membership Rate
if 'Loyalty Program Member' in df.columns:
    loyalty_rate = (df['Loyalty Program Member'].mean()) * 100
    print(f"Loyalty Program Membership Rate: {loyalty_rate:.2f}%")
else:
    print("Loyalty Program Membership Rate: Skipped (missing 'Loyalty Program Member')")

# Customer Segmentation by Value (Pie Chart)
if 'CLV' in df.columns:
    clv_segments = pd.qcut(df['CLV'], q=4, labels=['Low', 'Medium', 'High', 'Very High']).value_counts()
    plt.figure(figsize=(8, 8))
    plt.pie(clv_segments, labels=clv_segments.index, autopct='%1.1f%%', colors=sns.color_palette('muted'))
    plt.title("Customer Segmentation by Value")
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '6_customer_segmentation_by_value.png'))
    plt.close()
else:
    print("Customer Segmentation by Value: Skipped (missing 'CLV')")

# Engagement by Customer Type (Bar Chart)
if all(col in df.columns for col in ['Loyalty Program Member', 'Engagement Score']):
    engagement_by_type = df.groupby('Loyalty Program Member')['Engagement Score'].mean()
    plt.figure(figsize=(8, 6))
    engagement_by_type.plot(kind='bar', color='cyan')
    plt.title("Engagement by Customer Type")
    plt.xlabel("Loyalty Program Member")
    plt.ylabel("Avg. Engagement Score")
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '6_engagement_by_customer_type.png'))
    plt.close()
else:
    print("Engagement by Customer Type: Skipped (missing required columns)")

# CLV by Consumer Segment (Line Chart)
if all(col in df.columns for col in ['Transaction Date', 'CLV', 'Loyalty Program Member']):
    clv_trend = df.groupby(['Transaction Date', 'Loyalty Program Member'])['CLV'].mean().unstack()
    plt.figure(figsize=(10, 6))
    clv_trend.plot(kind='line', marker='o')
    plt.title("CLV by Consumer Segment")
    plt.xlabel("Date")
    plt.ylabel("CLV")
    plt.legend(title="Loyalty Program Member")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '6_clv_by_consumer_segment.png'))
    plt.close()
else:
    print("CLV by Consumer Segment: Skipped (missing required columns)")

# 7️⃣ Channel Performance & Sales Conversion
print("\n7️⃣ Channel Performance & Sales Conversion")
print("Card KPIs:")
# Highest Cart Abandonment Channel
if all(col in df.columns for col in ['Channel', 'Cart Abandoned']):
    cart_abandonment_by_channel = df.groupby('Channel')['Cart Abandoned'].mean() * 100
    highest_channel = cart_abandonment_by_channel.idxmax()
    print(f"Highest Cart Abandonment Channel: {highest_channel} ({cart_abandonment_by_channel[highest_channel]:.2f}%)")
else:
    print("Highest Cart Abandonment Channel: Skipped (missing required columns)")

# Channel with Most Customer Complaints
if all(col in df.columns for col in ['Channel', 'Support Queries']):
    complaints_by_channel = df.groupby('Channel')['Support Queries'].count()
    max_channel = complaints_by_channel.idxmax() if not complaints_by_channel.empty else "N/A"
    print(f"Channel with Most Customer Complaints: {max_channel}")
else:
    print("Channel with Most Customer Complaints: Skipped (missing required columns)")

# Channel with Declining Sales - Newly calculated
if all(col in df.columns for col in ['Channel', 'Revenue', 'Transaction Date']):
    channel_trends = df.groupby(['Channel', 'Transaction Date'])['Revenue'].sum().unstack()
    slopes = channel_trends.pct_change(axis=1).mean(axis=1)
    declining_channel = slopes.idxmin() if not slopes.empty else "N/A"
    decline_rate = slopes.min() * 100 if not slopes.empty else 0
    print(f"Channel with Declining Sales: {declining_channel} ({decline_rate:.2f}% avg. monthly decline)")
else:
    print("Channel with Declining Sales: Skipped (missing required columns)")

# Avg. Order Processing Time by Channel
if all(col in df.columns for col in ['Channel', 'Order Fulfillment Time']):
    avg_processing_time_by_channel = df.groupby('Channel')['Order Fulfillment Time'].mean().apply(
        lambda x: x.total_seconds() / 3600 if pd.notnull(x) else 0
    )
    print("Avg. Order Processing Time by Channel (hours):", avg_processing_time_by_channel.to_dict())
else:
    print("Avg. Order Processing Time by Channel: Skipped (missing required columns)")

# Fulfillment Delays by Channel - Newly calculated
if all(col in df.columns for col in ['Channel', 'Delivery Delays']):
    delays_by_channel = df.groupby('Channel')['Delivery Delays'].mean() * 100
    print("Fulfillment Delays by Channel (%):", delays_by_channel.to_dict())
else:
    print("Fulfillment Delays by Channel: Skipped (missing required columns)")

# Conversion Rate by Channel (Column Chart)
if all(col in df.columns for col in ['Channel', 'Payments Successful', 'Page Views']):
    conversion_rate_by_channel = df.groupby('Channel').apply(
        lambda x: min(100, (x['Payments Successful'].sum() / x['Page Views'].sum()) * 100) if x['Page Views'].sum() > 0 else 0
    )
    plt.figure(figsize=(10, 6))
    conversion_rate_by_channel.plot(kind='bar', color='lightblue')
    plt.title("Conversion Rate by Channel")
    plt.xlabel("Channel")
    plt.ylabel("Conversion Rate (%)")
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '7_conversion_rate_by_channel.png'))
    plt.close()
else:
    print("Conversion Rate by Channel: Skipped (missing required columns)")

# Channel-Specific NPS Score (Stacked Bar Chart)
if all(col in df.columns for col in ['Channel', 'Survey Response']):
    nps_by_channel = df.groupby('Channel')['Survey Response'].value_counts().unstack(fill_value=0)
    plt.figure(figsize=(10, 6))
    nps_by_channel.plot(kind='bar', stacked=True, cmap='viridis')
    plt.title("Channel-Specific NPS Score")
    plt.xlabel("Channel")
    plt.ylabel("Count")
    plt.legend(title="Response")
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '7_channel_specific_nps_score.png'))
    plt.close()
else:
    print("Channel-Specific NPS Score: Skipped (missing required columns)")

# Customer Complaints Trend by Channel (Bar Chart)
if all(col in df.columns for col in ['Transaction Date', 'Channel', 'Support Queries']):
    complaints_trend = df.groupby(['Transaction Date', 'Channel'])['Support Queries'].count().unstack(fill_value=0)
    plt.figure(figsize=(12, 6))
    complaints_trend.plot(kind='bar', cmap='plasma')
    plt.title("Customer Complaints Trend by Channel")
    plt.xlabel("Date")
    plt.ylabel("Complaints")
    plt.legend(title="Channel")
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '7_customer_complaints_trend_by_channel.png'))
    plt.close()
else:
    print("Customer Complaints Trend by Channel: Skipped (missing required columns)")

# 8️⃣ Marketing & Campaign Effectiveness
print("\n8️⃣ Marketing & Campaign Effectiveness")
print("Card KPIs:")
# Ad Click-Through Rate (CTR)
if all(col in df.columns for col in ['Ad Clicks', 'Impressions']):
    ctr = min(100, (df['Ad Clicks'].sum() / df['Impressions'].sum()) * 100) if df['Impressions'].sum() > 0 else 0
    print(f"Ad Click-Through Rate (CTR): {ctr:.2f}%")
else:
    print("Ad Click-Through Rate (CTR): Skipped (missing required columns)")

# Marketing ROI (Per Campaign) - Newly calculated
if all(col in df.columns for col in ['Revenue', 'Marketing Cost', 'Referral Source']):
    campaign_data = df.groupby('Referral Source').agg({'Revenue': 'sum', 'Marketing Cost': 'sum'})
    campaign_data['Baseline_Revenue'] = campaign_data['Revenue'].mean() * 0.5  # Assume 50% baseline
    campaign_data['ROI'] = (campaign_data['Revenue'] - campaign_data['Baseline_Revenue'] - campaign_data['Marketing Cost']) / campaign_data['Marketing Cost']
    print("Marketing ROI (Per Campaign):", campaign_data['ROI'].to_dict())
else:
    print("Marketing ROI (Per Campaign): Skipped (missing required columns)")

# Engagement Drop in Email Campaigns
if all(col in df.columns for col in ['Transaction Date', 'Newsletter Opened']):
    email_engagements = cache['by_date']['Newsletter Opened'].sum()
    if len(email_engagements) > 1:
        engagement_drop = ((email_engagements.iloc[0] - email_engagements.iloc[-1]) / email_engagements.iloc[0]) * 100 if email_engagements.iloc[0] > 0 else 0
    else:
        engagement_drop = 0
    print(f"Engagement Drop in Email Campaigns: {engagement_drop:.2f}%")
else:
    print("Engagement Drop in Email Campaigns: Skipped (missing required columns)")

# % of Traffic from Paid vs. Organic
if all(col in df.columns for col in ['Referral Source', 'Page Views']):
    paid_sources = ['Ads', 'Email', 'Social Media']
    paid_traffic = df[df['Referral Source'].isin(paid_sources)]['Page Views'].sum()
    total_traffic = df['Page Views'].sum()
    paid_traffic_percentage = min(100, (paid_traffic / total_traffic) * 100) if total_traffic > 0 else 0
    organic_traffic_percentage = min(100, 100 - paid_traffic_percentage)
    print(f"% of Traffic from Paid: {paid_traffic_percentage:.2f}%")
    print(f"% of Traffic from Organic: {organic_traffic_percentage:.2f}%")
else:
    print("% of Traffic from Paid vs. Organic: Skipped (missing required columns)")

# Ad Spend vs. Conversions (Scatter Plot) - Newly calculated
if all(col in df.columns for col in ['Marketing Cost', 'Payments Successful', 'Page Views']):
    ad_spend_vs_conv = df.groupby('Referral Source').agg({
        'Marketing Cost': 'sum',
        'Payments Successful': 'sum',
        'Page Views': 'sum'
    })
    ad_spend_vs_conv['Conversion_Rate'] = (ad_spend_vs_conv['Payments Successful'] / ad_spend_vs_conv['Page Views']) * 100
    plt.figure(figsize=(10, 6))
    plt.scatter(ad_spend_vs_conv['Marketing Cost'], ad_spend_vs_conv['Conversion_Rate'], s=100, c='darkgreen')
    for i, txt in enumerate(ad_spend_vs_conv.index):
        plt.annotate(txt, (ad_spend_vs_conv['Marketing Cost'].iloc[i], ad_spend_vs_conv['Conversion_Rate'].iloc[i]))
    plt.title("Ad Spend vs. Conversions")
    plt.xlabel("Marketing Cost")
    plt.ylabel("Conversion Rate (%)")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '8_ad_spend_vs_conversions.png'))
    plt.close()
else:
    print("Ad Spend vs. Conversions: Skipped (missing required columns)")

# Channel-Specific Marketing ROI (Stacked Bar Chart) - Newly calculated
if all(col in df.columns for col in ['Channel', 'Revenue', 'Marketing Cost']):
    channel_roi = df.groupby('Channel').agg({'Revenue': 'sum', 'Marketing Cost': 'sum'})
    channel_roi['Baseline_Revenue'] = channel_roi['Revenue'].mean() * 0.5
    channel_roi['ROI'] = (channel_roi['Revenue'] - channel_roi['Baseline_Revenue'] - channel_roi['Marketing Cost']) / channel_roi['Marketing Cost']
    plt.figure(figsize=(10, 6))
    channel_roi[['ROI']].plot(kind='bar', color='orchid')
    plt.title("Channel-Specific Marketing ROI")
    plt.xlabel("Channel")
    plt.ylabel("ROI")
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '8_channel_specific_marketing_roi.png'))
    plt.close()
else:
    print("Channel-Specific Marketing ROI: Skipped (missing required columns)")

# Consumer Engagement by Campaign Type (Heatmap)
if all(col in df.columns for col in ['Referral Source', 'Page Views', 'Impressions']):
    engagement_by_campaign = df.groupby('Referral Source').apply(
        lambda x: min(100, (x['Page Views'].sum() / x['Impressions'].sum()) * 100) if x['Impressions'].sum() > 0 else 0
    )
    plt.figure(figsize=(10, 6))
    sns.heatmap(engagement_by_campaign.to_frame().T, annot=True, cmap="YlGnBu", fmt=".1f")
    plt.title("Consumer Engagement by Campaign Type")
    plt.xlabel("Campaign Type")
    plt.ylabel("")
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '8_consumer_engagement_by_campaign_type.png'))
    plt.close()
else:
    print("Consumer Engagement by Campaign Type: Skipped (missing required columns)")

# Conversion Rate by Traffic Source (Column Chart)
if all(col in df.columns for col in ['Referral Source', 'Payments Successful', 'Page Views']):
    conversion_rate_by_source = df.groupby('Referral Source').apply(
        lambda x: min(100, (x['Payments Successful'].sum() / x['Page Views'].sum()) * 100) if x['Page Views'].sum() > 0 else 0
    )
    plt.figure(figsize=(10, 6))
    conversion_rate_by_source.plot(kind='bar', color='gold')
    plt.title("Conversion Rate by Traffic Source")
    plt.xlabel("Traffic Source")
    plt.ylabel("Conversion Rate (%)")
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '8_conversion_rate_by_traffic_source.png'))
    plt.close()
else:
    print("Conversion Rate by Traffic Source: Skipped (missing required columns)")

# 9️⃣ Logistics & Delivery Performance
print("\n9️⃣ Logistics & Delivery Performance")
print("Card KPIs:")
# Avg. Delivery Time
if 'Delivery Time' in df.columns:
    avg_delivery_time = df['Delivery Time'].mean().total_seconds() / 3600 if df['Delivery Time'].notnull().any() else 0
    print(f"Avg. Delivery Time (hours): {avg_delivery_time:.2f}")
else:
    print("Avg. Delivery Time: Skipped (missing 'Delivery Time')")

# Late Deliveries (%)
if 'Delivery Delays' in df.columns:
    late_deliveries = df['Delivery Delays'].sum()
    total_deliveries = len(df)
    late_deliveries_percentage = min(100, (late_deliveries / total_deliveries) * 100) if total_deliveries > 0 else 0
    print(f"Late Deliveries (%): {late_deliveries_percentage:.2f}%")
else:
    print("Late Deliveries (%): Skipped (missing 'Delivery Delays')")

# Top Reason for Delivery Delays
if 'Reason for Delivery Delays' in df.columns:
    top_delay_reason = df['Reason for Delivery Delays'].value_counts().idxmax() if not df['Reason for Delivery Delays'].value_counts().empty else "N/A"
    print(f"Top Reason for Delivery Delays: {top_delay_reason}")
else:
    print("Top Reason for Delivery Delays: Skipped (missing 'Reason for Delivery Delays')")

# Warehouse Processing Time
if 'Order Shipment Time' in df.columns:
    warehouse_processing_time = df['Order Shipment Time'].mean().total_seconds() / 3600 if df['Order Shipment Time'].notnull().any() else 0
    print(f"Warehouse Processing Time (hours): {warehouse_processing_time:.2f}")
else:
    print("Warehouse Processing Time: Skipped (missing 'Order Shipment Time')")

# Delivery Delay Trends Over Time (Line Chart)
if all(col in df.columns for col in ['Transaction Date', 'Delivery Delays']):
    delivery_delay_trend = cache['by_date']['Delivery Delays'].sum()
    plt.figure(figsize=(10, 6))
    delivery_delay_trend.plot(kind='line', marker='o', color='darkblue')
    plt.title("Delivery Delay Trends Over Time")
    plt.xlabel("Date")
    plt.ylabel("Delay Count")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '9_delivery_delay_trends_over_time.png'))
    plt.close()
else:
    print("Delivery Delay Trends Over Time: Skipped (missing required columns)")

# Warehouse Bottlenecks (Bar Chart)
if all(col in df.columns for col in ['Total Units', 'Transaction Date']):
    total_units = df['Total Units'].sum()
    time_period_days = (df['Transaction Date'].max() - df['Transaction Date'].min()).days
    throughput = total_units / time_period_days if time_period_days > 0 else 0
    plt.figure(figsize=(8, 6))
    plt.bar(['Total Units', 'Throughput'], [total_units, throughput], color=['blue', 'green'])
    plt.title("Warehouse Bottlenecks")
    plt.ylabel("Units")
    for i, v in enumerate([total_units, throughput]):
        plt.text(i, v + 0.5, f"{v:.2f}", ha='center')
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '9_warehouse_bottlenecks.png'))
    plt.close()
else:
    print("Warehouse Bottlenecks: Skipped (missing required columns)")

# Avg. Order Processing Time by Region (Bar Chart)
if all(col in df.columns for col in ['Region', 'Order Fulfillment Time']):
    processing_time_by_region = df.groupby('Region')['Order Fulfillment Time'].mean().apply(
        lambda x: x.total_seconds() / 3600 if pd.notnull(x) else 0
    )
    plt.figure(figsize=(10, 6))
    processing_time_by_region.plot(kind='bar', color='lightgreen')
    plt.title("Avg. Order Processing Time by Region")
    plt.xlabel("Region")
    plt.ylabel("Processing Time (hours)")
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '9_avg_order_processing_time_by_region.png'))
    plt.close()
else:
    print("Avg. Order Processing Time by Region: Skipped (missing required columns)")

# Return Due to Delivery Issues (Pie Chart)
if all(col in df.columns for col in ['Return Due to Delivery Issues', 'Return Requests']):
    delivery_returns = df[df['Return Due to Delivery Issues'] == 'Yes']['Return Requests'].sum()
    total_returns = df['Return Requests'].sum()
    plt.figure(figsize=(8, 8))
    plt.pie([delivery_returns, total_returns - delivery_returns], labels=['Delivery Issues', 'Other'], autopct='%1.1f%%', colors=['tomato', 'skyblue'])
    plt.title("Return Due to Delivery Issues")
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '9_return_due_to_delivery_issues.png'))
    plt.close()
else:
    print("Return Due to Delivery Issues: Skipped (missing required columns)")

# 🔟 Payment & Transaction Performance
print("\n🔟 Payment & Transaction Performance")
print("Card KPIs:")
# Transaction Failure Rate
if all(col in df.columns for col in ['Payments Attempted', 'Payments Successful']):
    transaction_failure_rate = min(100, ((df['Payments Attempted'].sum() - df['Payments Successful'].sum()) / df['Payments Attempted'].sum()) * 100) if df['Payments Attempted'].sum() > 0 else 0
    print(f"Transaction Failure Rate: {transaction_failure_rate:.2f}%")
else:
    print("Transaction Failure Rate: Skipped (missing required columns)")

# Most Common Failed Payment Method
if all(col in df.columns for col in ['Payment Method', 'Payments Attempted', 'Payments Successful']):
    failed_transactions = df[df['Payments Attempted'] > df['Payments Successful']].groupby('Payment Method').size()
    most_common_failed_method = failed_transactions.idxmax() if not failed_transactions.empty else "N/A"
    print(f"Most Common Failed Payment Method: {most_common_failed_method}")
else:
    print("Most Common Failed Payment Method: Skipped (missing required columns)")

# Chargeback Rate (Fraud Detection)
if all(col in df.columns for col in ['Number of Chargebacks', 'Payments Attempted']):
    chargeback_rate = min(100, (df['Number of Chargebacks'].sum() / df['Payments Attempted'].sum()) * 100) if df['Payments Attempted'].sum() > 0 else 0
    print(f"Chargeback Rate: {chargeback_rate:.2f}%")
else:
    print("Chargeback Rate: Skipped (missing required columns)")

# Fraudulent Transaction Detection - Newly calculated
if all(col in df.columns for col in ['Payments Attempted', 'Payments Successful', 'Transaction Amount']):
    df['Failure_Rate'] = (df['Payments Attempted'] - df['Payments Successful']) / df['Payments Attempted']
    avg_amount = df['Transaction Amount'].mean()
    fraud_flags = (df['Failure_Rate'] > 0.5) | (df['Transaction Amount'] > 3 * avg_amount)
    fraud_count = fraud_flags.sum()
    fraud_rate = (fraud_count / len(df)) * 100 if len(df) > 0 else 0
    print(f"Fraudulent Transaction Detection Rate: {fraud_rate:.2f}%")
else:
    print("Fraudulent Transaction Detection: Skipped (missing required columns)")

# Top Reasons for Payment Failure
if all(col in df.columns for col in ['Reasons for Payment Failure', 'Payments Attempted']):
    failure_reasons = df['Reasons for Payment Failure'].value_counts()
    print("Top Reasons for Payment Failure:", failure_reasons.to_dict())
else:
    print("Top Reasons for Payment Failure: Skipped (missing required columns)")

# Failed Transactions Over Time (Line Chart)
if all(col in df.columns for col in ['Transaction Date', 'Payments Attempted', 'Payments Successful']):
    failed_trend = cache['by_date'].apply(
        lambda x: (x['Payments Attempted'].sum() - x['Payments Successful'].sum()) / x['Payments Attempted'].sum() * 100 if x['Payments Attempted'].sum() > 0 else 0
    )
    plt.figure(figsize=(10, 6))
    failed_trend.plot(kind='line', marker='o', color='crimson')
    plt.title("Failed Transactions Over Time")
    plt.xlabel("Date")
    plt.ylabel("Failure Rate (%)")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '10_failed_transactions_over_time.png'))
    plt.close()
else:
    print("Failed Transactions Over Time: Skipped (missing required columns)")

# Payment Method vs. Success Rate (Stacked Bar Chart)
if all(col in df.columns for col in ['Payment Method', 'Payments Successful', 'Payments Attempted']):
    success_by_method = df.groupby('Payment Method').agg({'Payments Successful': 'sum', 'Payments Attempted': 'sum'})
    success_by_method['Failed'] = success_by_method['Payments Attempted'] - success_by_method['Payments Successful']
    plt.figure(figsize=(10, 6))
    success_by_method[['Payments Successful', 'Failed']].plot(kind='bar', stacked=True, color=['limegreen', 'red'])
    plt.title("Payment Method vs. Success Rate")
    plt.xlabel("Payment Method")
    plt.ylabel("Count")
    plt.legend(["Successful", "Failed"])
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '10_payment_method_vs_success_rate.png'))
    plt.close()
else:
    print("Payment Method vs. Success Rate: Skipped (missing required columns)")

# Chargeback Trend Analysis (Column Chart)
if all(col in df.columns for col in ['Transaction Date', 'Number of Chargebacks', 'Payments Attempted']):
    chargeback_trend = cache['by_date'].apply(
        lambda x: min(100, (x['Number of Chargebacks'].sum() / x['Payments Attempted'].sum()) * 100) if x['Payments Attempted'].sum() > 0 else 0
    )
    plt.figure(figsize=(10, 6))
    chargeback_trend.plot(kind='bar', color='indigo')
    plt.title("Chargeback Trend Analysis")
    plt.xlabel("Date")
    plt.ylabel("Chargeback Rate (%)")
    plt.tight_layout()
    plt.savefig(os.path.join(charts_folder, '10_chargeback_trend_analysis.png'))
    plt.close()
else:
    print("Chargeback Trend Analysis: Skipped (missing required columns)")

print("\nKPI calculation and chart saving completed.")

Validating dataset...

Calculating KPIs and saving charts...

Validating dataset...

Calculating KPIs and saving charts...

1️⃣ Customer Journey Analysis (Drop-off & Abandonment)
Card KPIs:
Cart Abandonment Rate: 53.77%
Checkout Abandonment Rate: 8.40%
Payment Failure Rate: 1.99%
Product Page Bounce Rate: 45.98%

2️⃣ Churn & Retention Analysis
Card KPIs:
Churn Rate by Consumer Segment: {0: 47.63292061179898, 1: 45.175169022741244}
Avg. Time Between Purchases: 385.10 days
Repeat Purchase Decline: Skipped (missing required columns)
Loyalty Program Drop-off Rate: 45.18%
Net Promoter Score (NPS): -68.10

3️⃣ Pricing Sensitivity & Discount Impact
Card KPIs:
Avg. Discount Utilization Rate: 100.00%
Price Sensitivity Index: -3.11
Elasticity of Demand: 3.11
Most Abandoned Discounted Product: Generic Product (0.25%)
Discount Effect on Repeat Purchase: Skipped (missing required columns)

4️⃣ Customer Satisfaction & Support Issues
Card KPIs:
Avg. Response Time (hours): 0.01
Avg. Resolution Time (h

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1200x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>